# 09 · BM25 evaluation (keyword baseline)

Implements **BM25** (Okapi BM25) over the movie corpus as a keyword-search baseline.  
Evaluated on the same 394-query test set used for the embedding-based models.

**Purpose:** Compare dense-retrieval results against a classical sparse-retrieval baseline.

## 0. Install dependencies

In [ ]:
!pip install rank_bm25 nltk

## 1. Imports

In [ ]:
import re
import string
import pickle
from difflib import SequenceMatcher

import nltk
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))
print("All imports OK")

## 2. Load movie corpus

In [ ]:
CORPUS_PATH = "../data/cleaned_movie_plots_v2.csv"

df_corpus = pd.read_csv(CORPUS_PATH)
print(f"Corpus size: {len(df_corpus):,} movies")
df_corpus.head(2)

## 3. Tokenise corpus and build BM25 index

Each document is the pre-built `embedding_text` field:  
`"Title: {title} Genre: {genre} Summary: {summary} Plot: {plot}"`

Tokenisation: lowercase → strip punctuation → remove stopwords → split on whitespace.

In [ ]:
def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # strip punctuation
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


print("Tokenising corpus...")
corpus_texts = df_corpus["embedding_text"].fillna("").tolist()
tokenized_corpus = [tokenize(doc) for doc in corpus_texts]

print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_corpus)
print(f"Index ready: {len(tokenized_corpus):,} documents")

## 4. Search helpers

In [ ]:
def search_bm25(query: str, top_k: int = 10) -> list[dict]:
    """Return top-k BM25 results as list of {title, score}."""
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {"title": df_corpus.iloc[idx]["title"], "score": float(scores[idx])}
        for idx in top_indices
    ]


def fuzzy_match(a: str, b: str, threshold: float = 0.85) -> tuple[bool, float]:
    ratio = SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()
    return ratio >= threshold, ratio


# Quick sanity check
sample = search_bm25("a shark terrorizes a beach town")
print("Sample query results:")
for r in sample:
    print(f"  {r['title']}  (score={r['score']:.3f})")

## 5. Load test set

In [ ]:
df_test = pd.read_csv("../data/eval/test_set_394.csv")
print(f"Test queries: {len(df_test)}")
df_test["query_type"].value_counts()

## 6. Run BM25 evaluation

In [ ]:
rows = []

for i, row in df_test.iterrows():
    results = search_bm25(row["query"], top_k=10)

    correct_rank = match_ratio = correct_score = None
    for rank, r in enumerate(results, 1):
        ok, ratio = fuzzy_match(r["title"], row["title"])
        if ok:
            correct_rank, match_ratio, correct_score = rank, ratio, r["score"]
            break

    rows.append({
        "title":            row["title"],
        "year":             row.get("release_year"),
        "genre":            row.get("genre"),
        "query":            row["query"],
        "query_type":       row["query_type"],
        "top1_result":      results[0]["title"] if results else None,
        "correct_rank":     correct_rank if correct_rank is not None else "not found",
        "match_ratio":      round(match_ratio, 3) if match_ratio else None,
        "hit@1":            int(correct_rank == 1) if correct_rank else 0,
        "hit@3":            int(correct_rank is not None and correct_rank <= 3),
        "hit@5":            int(correct_rank is not None and correct_rank <= 5),
        "hit@10":           int(correct_rank is not None and correct_rank <= 10),
        "reciprocal_rank":  (1 / correct_rank) if correct_rank else 0.0,
        "bm25_score":       correct_score or 0.0,
    })

    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(df_test)}")

results_df = pd.DataFrame(rows)
results_df.to_csv("../results/eval_394_bm25.csv", index=False)
print(f"\nSaved → ../results/eval_394_bm25.csv ({len(results_df)} rows)")

## 7. Overall metrics

In [ ]:
overall = {
    "Hit@1":  results_df["hit@1"].mean(),
    "Hit@3":  results_df["hit@3"].mean(),
    "Hit@5":  results_df["hit@5"].mean(),
    "Hit@10": results_df["hit@10"].mean(),
    "MRR":    results_df["reciprocal_rank"].mean(),
}
print("=== BM25 (Okapi BM25) ===")
for k, v in overall.items():
    print(f"  {k}: {v:.1%}")

## 8. Per query-type breakdown

In [ ]:
print("=== Per query type (Hit@1 / Hit@5 / Hit@10 / MRR) ===")
for qt in ["oracle", "conversational", "naturalistic", "vague"]:
    sub = results_df[results_df["query_type"] == qt]
    if len(sub):
        print(
            f"  {qt:>15s}: "
            f"H@1={sub['hit@1'].mean():.1%}  "
            f"H@5={sub['hit@5'].mean():.1%}  "
            f"H@10={sub['hit@10'].mean():.1%}  "
            f"MRR={sub['reciprocal_rank'].mean():.3f}  "
            f"(n={len(sub)})"
        )

## 9. Summary table

In [ ]:
summary = results_df.groupby("query_type").agg(
    n=("hit@1", "count"),
    Hit_at_1=("hit@1", "mean"),
    Hit_at_3=("hit@3", "mean"),
    Hit_at_5=("hit@5", "mean"),
    Hit_at_10=("hit@10", "mean"),
    MRR=("reciprocal_rank", "mean"),
).reset_index()

for col in ["Hit_at_1", "Hit_at_3", "Hit_at_5", "Hit_at_10", "MRR"]:
    summary[col] = summary[col].map(lambda x: f"{x:.1%}")

display(summary)

## 10. Rank distribution — how far down does BM25 find the correct movie?

In [ ]:
numeric_ranks = pd.to_numeric(results_df["correct_rank"], errors="coerce")
found_mask = numeric_ranks.notna()

print(f"Found in top-10: {found_mask.sum()} / {len(results_df)} ({found_mask.mean():.1%})")
print(f"Not found:       {(~found_mask).sum()} / {len(results_df)} ({(~found_mask).mean():.1%})")
print()
print("Rank distribution (when found):")
print(numeric_ranks[found_mask].value_counts().sort_index().to_string())